In [3]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"
pio.templates.default = "ggplot2"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

from glob import glob

csvs = glob("*.csv")
dfs = [pd.read_csv(csv) for csv in csvs]

for i, df in enumerate(dfs):
    df.drop(columns=["Unnamed: 0"], inplace=True, errors='ignore')
    df["dataset"] = csvs[i].split("-")[0]
    df["model"] = csvs[i].split("-")[1]
    df["epoch"] = int(csvs[i].split("-b")[-1].split(".csv")[0])
    df["name"] = csvs[i].split(".csv")[0]
    
data = pd.concat(dfs, ignore_index=True)
data.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,Simplification Threshold,Count,dataset,model,epoch,name
0,0.000000e+00,18,bert,simpl,4,bert-simpl-b4
1,0.000000e+00,17,bert,simpl,4,bert-simpl-b4
2,1.192020e-07,16,bert,simpl,4,bert-simpl-b4
3,1.192020e-07,15,bert,simpl,4,bert-simpl-b4
4,1.311215e-06,14,bert,simpl,4,bert-simpl-b4


In [ ]:
ymark = 9

In [11]:
combined = make_subplots(rows=1, cols=1)
leg = dict(x=0.98, y=0.98, xanchor="right", yanchor="top", bgcolor="rgba(255, 255, 255, 0.6)")
combined.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=300*2, height=180*2, font=dict(size=16), legend=leg)
combined.update_xaxes(title_text="Simplification Threshold", title_standoff=18, automargin=True, range=[0, data["Simplification Threshold"].max()])
combined.update_yaxes(title_text="#Valleys", type="log", title_standoff=18, automargin=True)

for (model, ds, epoch), df in data.groupby(["model", "dataset", "epoch"]):
	fig = make_subplots(rows=1, cols=1)
	df = df.sort_values(by="Count", ascending=False, inplace=False)
	name = df["name"].unique()[0]

	tr = go.Scatter(
		x=df["Simplification Threshold"],
		y=df["Count"],
		mode="lines",
		name="Base" if epoch == 0 else f"Fine-tuned",
		line=dict(shape="hv"),
		showlegend=True
	)

	fig.add_trace(tr, row=1, col=1)
	combined.add_trace(tr, row=1, col=1)
 
	fig.add_trace(
		go.Scatter(
			x=[-100, 10000],
			y=[ymark, ymark],
			mode="lines",
			name=f"#classes",
			line=dict(color="forestgreen", width=2, dash="dash"),
			opacity=0.5,
   			showlegend=False
		),
		row=1,
		col=1,
	)

	fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=300*2, height=180*2, font=dict(size=16), legend=dict(x=0.98, y=0.98, xanchor="right", yanchor="top", bgcolor="rgba(255, 255, 255, 0.72)"))
	fig.update_xaxes(title_text="Simplification Threshold", title_standoff=18, automargin=True, range=[0, df["Simplification Threshold"].max()])
	fig.update_yaxes(title_text="#Valleys", type="log", title_standoff=18, automargin=True)
	fig.write_image(f"plots/{name}.png", scale=4)

combined.show()
combined.write_image("plots/combined.png", scale=4)

In [18]:
for (model, ds, epoch), df in data.groupby(["model", "dataset", "epoch"]):
	fig = make_subplots(rows=1, cols=1)
	df = df.sort_values(by="Count", ascending=False, inplace=False)
	name = df["name"].unique()[0]

	fig.add_trace(
		go.Scatter(
			x=df["Simplification Threshold"],
			y=df["Count"],
			mode="lines",
			name=ds,
			line=dict(shape="hv"),
			showlegend=False
  		),
		row=1,
		col=1,
	)
 
	fig.add_trace(
		go.Scatter(
			x=[-100, 10000],
			y=[ymark, ymark],
			mode="lines",
			name=f"#classes",
			line=dict(color="forestgreen", width=2, dash="dash"),
			opacity=0.5,
			showlegend=False
		),
		row=1,
		col=1,
	)

	fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=300*2, height=180*2, font=dict(size=16))
	fig.update_xaxes(title_text="Simplification Threshold", title_standoff=18, automargin=True, range=[0, df["Simplification Threshold"].max()])
	fig.update_yaxes(title_text="#Valleys", title_standoff=18, automargin=True)
	fig.write_image(f"plots/{name}_nolog.png", scale=8)

In [35]:
figs = {}

# if I only want to fill out figs
CAPTURE_MODE = True

for j, (model, df) in enumerate(data.groupby("model")):
	dses = df["dataset"].unique()
	fig = make_subplots(rows=1, cols=1, shared_yaxes=True)
	figs[model] = {"traces": [], "xmax": []}

	for i, (ds, df) in enumerate(df.groupby(["dataset"])):		
		df = df.sort_values(by="Number of Minima", ascending=False, inplace=False)

		showlegend = True
  
		if CAPTURE_MODE and j != 0:
			showlegend = False
		
		trace = go.Scatter(
			x=df["Simplification Threshold"],
			y=df["Number of Minima"],
			mode="lines",
			line=dict(shape="hv", color=color_seq[i], width=3),
			name=ds[0],
			opacity=0.6,
			showlegend=showlegend
		)

		figs[model]["traces"].append(trace)

		fig.add_trace(trace, row=1, col=1)
  
		max_x = df["Simplification Threshold"].max()
		if model == "wres":
			max_x = 0.001
		elif model == "vgg":
			max_x = 0.004
		
		figs[model]["xmax"].append(max_x)
		fig.update_xaxes(range=[0, max_x], row=1, col=i+1)
  
	fig.update_annotations(font_size=16)
	fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=300*2, height=180*2, font=dict(size=14), legend=dict(
     xanchor="right", yanchor="top", x=0.98, y=0.98, bgcolor="rgba(255,255,255,0.5)", font=dict(size=14)
    ))
	fig.update_xaxes(title_text="Simplification Threshold", title_standoff=18, automargin=True)
	fig.update_yaxes(title_text="#Valleys", type="log", title_standoff=18, automargin=True)
	if not CAPTURE_MODE:
		fig.write_image(f"plots/{model}.png", scale=8)

In [21]:
models_to_idx = {mod: i for i, mod in enumerate(data["model"].unique())}

In [49]:
fig = make_subplots(rows=1, cols=1, shared_yaxes=True)

no_wres = data[data["model"] != "wres"]
for i, ((model, ds), rows) in enumerate(no_wres.groupby(["model", "dataset"])):
	dash = "solid" if ds == "cifar" else "dash"
 
	trace = go.Scatter(x = rows["Simplification Threshold"], y = rows["Number of Minima"], mode="lines", 
                    line=dict(shape="vh", dash=dash), name=f"{model}", line_color=color_seq[models_to_idx[model]], opacity=0.8)
	fig.add_trace(trace, row=1, col=1)
 
fig.update_layout(margin=dict(l=0, r=0, t=30, b=0), width=300*2, height=180*2, font=dict(size=16), showlegend=True)
fig.update_xaxes(title_text="Simplification Threshold", title_standoff=18, automargin=True, range=[0, 0.02])
fig.update_yaxes(title_text="#Valleys", type="log", title_standoff=18, automargin=True)
fig.show()
	

In [36]:
titlemap = {
	"resnet": "ResNet",
	"vgg": "VGG",
	"densenet": "DenseNet",
}

In [41]:
df_no_wres = data[data["model"] != "wres"]

dses = df_no_wres["dataset"].unique()
fig = make_subplots(rows=1, cols=3, shared_yaxes=True, subplot_titles=[titlemap[model] for model in df_no_wres["model"].unique()])

for i, (model, df) in enumerate(df_no_wres.groupby("model")):
	df = df.sort_values(by="Number of Minima", ascending=False, inplace=False)

	fig.add_traces(figs[model]["traces"], rows=1, cols=i+1)
	fig.update_xaxes(range=[0, max(figs[model]["xmax"])], row=1, col=i+1)
 
fig.update_annotations(font_size=16)
fig.update_layout(margin=dict(l=0, r=0, t=30, b=0), width=700*2, height=180*2, font=dict(size=14), legend=dict(
	xanchor="right", yanchor="top", x=0.98, y=0.98, bgcolor="rgba(255,255,255,0.5)", font=dict(size=14)
))
fig.update_xaxes(title_text="Simplification Threshold", title_standoff=18, automargin=True)
fig.update_yaxes(type="log", title_standoff=18, automargin=True)
fig.update_yaxes(title_text="#Valleys", row=1, col=1)
fig.write_image(f"plots/combined.png", scale=8)
# fig.show()

In [13]:
model = "resnet"
ds = "cifar"
df = data[(data["model"] == model) & (data["dataset"] == ds)]

fig = make_subplots(rows=1, cols=1, shared_yaxes=True)

df = df.sort_values(by="Number of Minima", ascending=False, inplace=False)

fig.add_trace(
	go.Scatter(
		x=df["Simplification Threshold"],
		y=df["Number of Minima"],
		mode="lines",
		line=dict(shape="hv", color=color_seq[i], width=3),
		name=ds,
		opacity=1
	),
	row=1,
	col=1
)

fig.add_trace(
	go.Scatter(
		x = [0.0004965, 0.0004965],
		y = [1, 290],
		mode="lines",
		line=dict(color="black", width=2, dash="dash"),
	)
)

  
fig.update_annotations(font_size=16)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=300*2, height=180*2, font=dict(size=14), showlegend=False)
fig.add_vline(x=0.0004965, line_dash="dash", line=dict(color="#C20000"))
fig.update_xaxes(title_text="Simplification", title_standoff=18, automargin=True, range=[0, 0.015])
fig.update_yaxes(title_text="#Valleys", type="log", title_standoff=18, automargin=True, range=[0, 2.5], nticks=5)
fig.write_image(f"plots/cifar-resnet-b138-interface.png", scale=8)